In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.linear_model import LogisticRegression




In [2]:
def extract_features(
    img_path,
    size=(224, 224),  # larger resize for finer details
    rgb_hist_bins=64,  # more bins → richer colour description
    hsv_hist_bins=64,
    edge_bins=32,  # finer edge magnitude histogram
):
    """
    Extract a richer colour‑texture descriptor:
    - Resize image for finer histograms.
    - Compute mean/std for RGB and HSV channels.
    - Compute per‑channel histograms for RGB and HSV (density normalised).
    - Compute a simple edge‑magnitude histogram on the grayscale image.
    - Concatenate all parts and L2‑normalise.
    """
    img = Image.open(img_path).convert("RGB")
    img = img.resize(size)
    arr_rgb = np.asarray(img).astype(np.float32)  # (H, W, 3)

    mean_rgb = arr_rgb.mean(axis=(0, 1))
    std_rgb = arr_rgb.std(axis=(0, 1))

    rgb_hist = []
    for c in range(3):
        hist, _ = np.histogram(
            arr_rgb[:, :, c], bins=rgb_hist_bins, range=(0, 256), density=True
        )
        rgb_hist.append(hist)
    rgb_hist = np.concatenate(rgb_hist)  # (rgb_hist_bins*3,)

    img_hsv = img.convert("HSV")
    arr_hsv = np.asarray(img_hsv).astype(np.float32)

    mean_hsv = arr_hsv.mean(axis=(0, 1))
    std_hsv = arr_hsv.std(axis=(0, 1))

    hsv_hist = []
    for c in range(3):
        hist, _ = np.histogram(
            arr_hsv[:, :, c], bins=hsv_hist_bins, range=(0, 256), density=True
        )
        hsv_hist.append(hist)
    hsv_hist = np.concatenate(hsv_hist)  # (hsv_hist_bins*3,)

    img_gray = img.convert("L")
    arr_gray = np.asarray(img_gray).astype(np.float32)
    gy, gx = np.gradient(arr_gray)
    edge_magnitude = np.sqrt(gx**2 + gy**2)
    edge_hist, _ = np.histogram(
        edge_magnitude, bins=edge_bins, range=(0, 255), density=True
    )

    feat = np.concatenate(
        [mean_rgb, std_rgb, rgb_hist, mean_hsv, std_hsv, hsv_hist, edge_hist]
    )
    norm = np.linalg.norm(feat)
    if norm > 0:
        feat = feat / norm
    return feat




In [3]:
work_dir = "/kaggle/input/cassava-leaf-disease-classification/"
train_dir = os.path.join(work_dir, "train_images")
test_dir = os.path.join(work_dir, "test_images")
train_df = pd.read_csv(os.path.join(work_dir, "train.csv"))
sample_sub_path = os.path.join(work_dir, "sample_submission.csv")

# Containers for centroid baseline (kept for compatibility) and for supervised training
class_sums = {}
class_counts = {}

X_train = []  # feature vectors for logistic regression
y_train = []  # corresponding labels

for _, row in train_df.iterrows():
    label = int(row["label"])
    img_path = os.path.join(train_dir, row["image_id"])
    try:
        feat = extract_features(img_path)
        # update centroid accumulators
        class_sums[label] = class_sums.get(label, np.zeros_like(feat)) + feat
        class_counts[label] = class_counts.get(label, 0) + 1
        # store for supervised model
        X_train.append(feat)
        y_train.append(label)
    except Exception:
        continue

# Build normalized centroids (still unused by the new classifier)
centroids = {}
for lbl in class_sums:
    centroid = class_sums[lbl] / class_counts[lbl]
    norm = np.linalg.norm(centroid)
    if norm > 0:
        centroid = centroid / norm
    centroids[lbl] = centroid

# Train a multinomial logistic‑regression classifier on the extracted features
logreg = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=200,
    n_jobs=-1,
    verbose=0,
)
logreg.fit(np.stack(X_train), np.array(y_train))



/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=200, multi_class='multinomial', n_jobs=-1)

In [4]:
sample_sub = pd.read_csv(sample_sub_path)
test_images = sample_sub["image_id"].tolist()  # deterministic order as required
predictions = []

for img_name in test_images:
    img_path = os.path.join(test_dir, img_name)
    try:
        feat = extract_features(img_path)
        pred_label = int(logreg.predict(feat.reshape(1, -1))[0])
    except Exception:
        pred_label = 0  # fallback if image cannot be processed
    predictions.append(pred_label)

submission = pd.DataFrame({"image_id": test_images, "label": predictions})
submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")



Submission saved to /kaggle/working/submission.csv


In [5]:
submission.head()

,image_id,label
0,1234294272.jpg,3
1,1234332763.jpg,3
2,1234375577.jpg,3
3,1234555380.jpg,3
4,1234571117.jpg,3
